# coding: utf-8

# # Aula Completa: Funções de Janela (Window Functions) com PySpark
# 
# As Funções de Janela no PySpark permitem que você execute cálculos em um conjunto de linhas (uma "janela") que estão de alguma forma relacionadas à linha atual. Diferentemente das agregações `groupBy`, as funções de janela não colapsam as linhas do DataFrame original; elas retornam um valor para cada linha com base na janela definida.
# 
# ## Por que usar Funções de Janela?
# 
# - **Cálculos Comparativos**: Comparar uma linha com outras dentro do mesmo grupo (ex: salário de um funcionário vs. média salarial do departamento).
# - **Rankings**: Classificar linhas dentro de partições (ex: top N produtos por vendas em cada categoria).
# - **Análises de Séries Temporais**: Calcular médias móveis, diferenças em relação a períodos anteriores, etc.
# - **Manutenção da Granularidade**: Obter resultados agregados sem perder os detalhes das linhas individuais.
# 
# ## Componentes Principais de uma Função de Janela
# 
# Para usar funções de janela, você primeiro define uma "especificação de janela" usando `Window` do módulo `pyspark.sql.window`. Os componentes chave são:
# 
# 1.  **`partitionBy()`**: Divide as linhas do DataFrame em partições. A função de janela é aplicada independentemente a cada partição. É semelhante ao `GROUP BY`.
# 2.  **`orderBy()`**: Ordena as linhas dentro de cada partição. Isso é crucial para funções de ranking e funções analíticas como `lag()` e `lead()`.
# 3.  **`rowsBetween()` / `rangeBetween()`** (Frame da Janela): Define o subconjunto de linhas dentro de uma partição a ser usado para o cálculo em relação à linha atual (ex: "as 2 linhas anteriores e a linha atual"). Para simplificar, vamos focar mais no `partitionBy` e `orderBy` nesta aula inicial.
# 
# ## Vamos Começar: Configuração do SparkSession

# In[1]:


# Importando as bibliotecas necessárias
from pyspark.sql import SparkSession, Window
import pyspark.sql.functions as F

# Criando uma SparkSession (ponto de entrada para programar com Spark)
# É uma boa prática configurar um nome para sua aplicação
spark = SparkSession.builder \
    .appName("AulaWindowFunctions") \
    .config("spark.ui.port", "4051") # Porta para UI do Spark, caso a 4040 esteja em uso
    .getOrCreate()

# Verificando a versão do Spark
print(f"Versão do Spark: {spark.version}")


# ## Criando um DataFrame de Exemplo
# 
# Vamos usar um DataFrame de exemplo com dados de vendas de produtos em diferentes categorias e datas.

# In[2]:


# Dados de exemplo: produto, categoria, data_venda, quantidade, valor_unitario
dados_vendas = [
    ("Produto A", "Eletrônicos", "2023-01-10", 10, 50.0),
    ("Produto B", "Eletrônicos", "2023-01-11", 5, 120.0),
    ("Produto C", "Livros", "2023-01-10", 20, 15.0),
    ("Produto A", "Eletrônicos", "2023-01-12", 8, 50.0),
    ("Produto D", "Casa", "2023-01-11", 12, 25.0),
    ("Produto B", "Eletrônicos", "2023-01-13", 7, 120.0),
    ("Produto C", "Livros", "2023-01-12", 15, 15.0),
    ("Produto E", "Livros", "2023-01-13", 10, 20.0),
    ("Produto D", "Casa", "2023-01-14", 10, 25.0)
]

# Definindo o schema (nomes das colunas e tipos de dados)
colunas_vendas = ["produto", "categoria", "data_venda", "quantidade", "valor_unitario"]

# Criando o DataFrame
df_vendas = spark.createDataFrame(dados_vendas, colunas_vendas)

# Calculando o valor total da venda para cada linha
df_vendas = df_vendas.withColumn("valor_total_venda", F.col("quantidade") * F.col("valor_unitario"))

# Convertendo a coluna de data para o tipo Date
df_vendas = df_vendas.withColumn("data_venda", F.to_date(F.col("data_venda")))

# Exibindo o DataFrame inicial
print("DataFrame de Vendas Original:")
df_vendas.show()


# ## 1. Funções de Agregação sobre Janelas
# 
# Podemos usar funções de agregação comuns (como `sum`, `avg`, `min`, `max`, `count`) com uma especificação de janela.
# 
# **Exemplo**: Calcular o total de vendas e a média do valor total de venda por categoria.

# In[3]:


# Definindo a especificação da janela: particionar por 'categoria'
# Para cada linha, a janela incluirá todas as linhas da mesma categoria
janela_categoria = Window.partitionBy("categoria")

# Calculando o total de vendas (valor_total_venda) por categoria
df_vendas_agg = df_vendas.withColumn(
    "total_vendas_categoria", 
    F.sum("valor_total_venda").over(janela_categoria)
)

# Calculando a média do valor total de venda por categoria
df_vendas_agg = df_vendas_agg.withColumn(
    "media_vendas_categoria",
    F.avg("valor_total_venda").over(janela_categoria)
)

# Calculando o número de vendas por categoria
df_vendas_agg = df_vendas_agg.withColumn(
    "num_vendas_categoria",
    F.count("*").over(janela_categoria) # Conta todas as linhas na partição
)

print("DataFrame com Agregações por Categoria (usando Window Functions):")
df_vendas_agg.orderBy("categoria", "produto").show(truncate=False)

# Note que todas as linhas originais são mantidas, e as novas colunas
# contêm os valores agregados repetidos para cada linha dentro da mesma categoria.


# ## 2. Funções de Ranking
# 
# Funções de ranking são usadas para classificar linhas dentro de uma partição. `orderBy` é essencial aqui.
# 
# -   `rank()`: Atribui um rank com "gaps" (saltos). Se houver empates, o próximo rank pula. (Ex: 1, 2, 2, 4)
# -   `dense_rank()`: Atribui um rank sem "gaps". Se houver empates, o próximo rank é consecutivo. (Ex: 1, 2, 2, 3)
# -   `row_number()`: Atribui um número sequencial único para cada linha dentro da partição, mesmo com empates (a ordem entre empates pode ser não determinística sem um critério de desempate).
# -   `ntile(n)`: Divide as linhas em `n` "baldes" (grupos) ordenados.
# 
# **Exemplo**: Rankear os produtos dentro de cada categoria pelo `valor_total_venda` (do maior para o menor).

# In[4]:


# Definindo a especificação da janela: particionar por 'categoria' e ordenar por 'valor_total_venda' descendentemente
janela_categoria_ordenada_valor = Window.partitionBy("categoria").orderBy(F.desc("valor_total_venda"))

# Aplicando as funções de ranking
df_vendas_ranked = df_vendas.withColumn(
    "rank_na_categoria",
    F.rank().over(janela_categoria_ordenada_valor)
).withColumn(
    "dense_rank_na_categoria",
    F.dense_rank().over(janela_categoria_ordenada_valor)
).withColumn(
    "row_number_na_categoria",
    F.row_number().over(janela_categoria_ordenada_valor)
)

print("DataFrame com Ranking de Produtos por Categoria (baseado no valor_total_venda):")
df_vendas_ranked.orderBy("categoria", "rank_na_categoria").show(truncate=False)


# ## 3. Funções Analíticas (Lag e Lead)
# 
# -   `lag(col, count, default)`: Acessa o valor de uma coluna da linha anterior (`count` linhas antes) dentro da partição ordenada.
# -   `lead(col, count, default)`: Acessa o valor de uma coluna da próxima linha (`count` linhas depois) dentro da partição ordenada.
# 
# `orderBy` é crucial para `lag` e `lead` para definir o que é "anterior" e "próximo".
# 
# **Exemplo**: Para cada venda de um produto, mostrar o valor da venda anterior do mesmo produto.

# In[5]:


# Definindo a especificação da janela: particionar por 'produto' e ordenar por 'data_venda'
# Isso garante que estamos olhando para vendas sequenciais do mesmo produto
janela_produto_ordenada_data = Window.partitionBy("produto").orderBy("data_venda")

# Calculando o valor da venda anterior para o mesmo produto
df_vendas_lag = df_vendas.withColumn(
    "valor_venda_anterior_produto",
    F.lag("valor_total_venda", 1).over(janela_produto_ordenada_data) # Pega o valor da linha anterior (offset 1)
)

# Calculando o valor da próxima venda para o mesmo produto
df_vendas_lead = df_vendas_lag.withColumn( # Podemos continuar do df anterior
    "valor_venda_proxima_produto",
    F.lead("valor_total_venda", 1).over(janela_produto_ordenada_data) # Pega o valor da próxima linha (offset 1)
)

print("DataFrame com Valor da Venda Anterior e Próxima por Produto:")
df_vendas_lead.orderBy("produto", "data_venda").show(truncate=False)

# Note que para a primeira venda de cada produto, 'valor_venda_anterior_produto' será null.
# E para a última venda de cada produto, 'valor_venda_proxima_produto' será null.
# Podemos fornecer um valor padrão para o lag/lead se quisermos.
# Ex: F.lag("valor_total_venda", 1, 0.0).over(janela_produto_ordenada_data)


# ## Exemplo Combinado: Análise de Vendas por Categoria e Data
# 
# Vamos calcular a soma acumulada (running total) de `valor_total_venda` para cada categoria, ordenado por data.

# In[6]:


# Definindo a especificação da janela: particionar por 'categoria' e ordenar por 'data_venda'
# O frame da janela padrão (quando orderBy é usado) é de `unboundedPreceding` (desde o início da partição)
# até `currentRow` (a linha atual). Isso é perfeito para somas acumuladas.
janela_categoria_data_acumulada = Window.partitionBy("categoria") \
                                       .orderBy("data_venda") \
                                       .rowsBetween(Window.unboundedPreceding, Window.currentRow)
                                       # rowsBetween define explicitamente o frame.
                                       # Para soma acumulada, se orderBy está presente,
                                       # o frame padrão já é unboundedPreceding to currentRow.
                                       # Mas é bom ser explícito para clareza.

df_soma_acumulada = df_vendas.withColumn(
    "soma_acumulada_categoria_por_data",
    F.sum("valor_total_venda").over(janela_categoria_data_acumulada)
)

print("DataFrame com Soma Acumulada de Vendas por Categoria e Data:")
df_soma_acumulada.orderBy("categoria", "data_venda").show(truncate=False)


# ## Conclusão
# 
# As Funções de Janela são uma ferramenta extremamente versátil no PySpark para realizar análises de dados complexas. Elas permitem:
# 
# -   Calcular agregações sem perder a granularidade dos dados.
# -   Criar rankings e numerações dentro de grupos.
# -   Analisar dados sequenciais (como séries temporais) com `lag` e `lead`.
# -   Calcular somas acumuladas, médias móveis e muito mais.
# 
# A chave é entender como definir corretamente a especificação da janela (`partitionBy`, `orderBy` e, opcionalmente, o frame com `rowsBetween` ou `rangeBetween`) para obter o resultado desejado.
# 
# ## Próximos Passos
# 
# -   Experimente com diferentes funções de janela (`cume_dist`, `percent_rank`, `ntile`).
# -   Explore o uso de `rowsBetween` e `rangeBetween` para definir frames de janela mais específicos (ex: média móvel dos últimos 3 dias).
# -   Combine funções de janela com outras transformações do Spark para construir pipelines de análise de dados robustos.

# In[7]:


# Não se esqueça de parar a SparkSession quando terminar de usá-la
# para liberar os recursos.
spark.stop()
print("Sessão Spark finalizada.")
